## Start

In [ ]:
import os
from dotenv import load_dotenv
import pandas as pd
import plotly.express as px
import datetime
import plotly.graph_objects as go


load_dotenv()
ticker = ''
path_stockdata = os.path.join(os.path.join(os.environ.get('judgement_day'), 'Data--StockWatchList'), ticker)
path_analysis_csv = os.path.join(path_stockdata, f'{ticker}--Analysis_DYT-s1v1.csv')

today = datetime.date.today()
cutoff = today.year - 20
cutoff10 = today.year - 10
cutoff5 = today.year - 5
cutoff3 = today.year - 3


## Data Frame Imports

Notable Cleaning
- Convert string dates to datetime
- Filter down to last 20 years of data
- Filter out current year in dvtagr_df as it will have incomplete data
- Convert decimal yields to numeric percentage for easy viewing

In [ ]:
%%capture

dyt_df0 = pd.read_csv(os.path.join(path_stockdata, f'{ticker}--DivPrice_History-s1v1.csv'), index_col=0)
dyt_df0['Date'] = pd.to_datetime(dyt_df0['Date'])
dyt_df1 = dyt_df0[dyt_df0['Date'].dt.year >= cutoff]
dyt_df1['FwdDiv%'] = dyt_df0['FwdDivYield']


dytagr_df0 = pd.read_csv(os.path.join(path_stockdata, f'{ticker}--Aggregate_Cy_DivPrice_History-s1v1.csv'), index_col=0)
dytagr_df1 = dytagr_df0.query('DateCy >= @cutoff and DateCy < @today.year')
dytagr_df1['Min%'] = round(dytagr_df1['DivYieldMin'] * 100, 2)
dytagr_df1['Max%'] = round(dytagr_df1['DivYieldMax'] * 100, 2)
dytagr_df1['Mean%'] = round(dytagr_df1['DivYieldMean'] * 100, 2)
dytagr_df1['Median%'] = round(dytagr_df1['DivYieldMedian'] * 100, 2)





In [ ]:
print('dyt_df1 Row Count:', len(dyt_df1))
dyt_df1

In [ ]:
print('dytagr_df1 Row Count:', len(dytagr_df1))
dytagr_df1

## View Aggregate Yield Data By year

In [ ]:
dytagr_df1[['Min%', 'Max%', 'Mean%', 'Median%']]

## Calc Mean, Median & Rolling

In [ ]:
mean20 = dyt_df1['FwdDiv%'].mean()
median20 = dyt_df1['FwdDiv%'].median()

dyt_df1_l10 = dyt_df1[dyt_df1['Date'].dt.year >= cutoff10]
mean10 = dyt_df1_l10['FwdDiv%'].mean()
median10 = dyt_df1_l10['FwdDiv%'].median()

dyt_df1_l5 = dyt_df1[dyt_df1['Date'].dt.year >= cutoff5]
mean5 = dyt_df1_l5['FwdDiv%'].mean()
median5 = dyt_df1_l5['FwdDiv%'].median()

dyt_df1_l3 = dyt_df1[dyt_df1['Date'].dt.year >= cutoff3]
mean3 = dyt_df1_l3['FwdDiv%'].mean()
median3 = dyt_df1_l3['FwdDiv%'].median()

print(f'Mean20Yr Div: {round(mean20 * 100, 2)}%')
print(f'Median20Yr Div: {round(median20 * 100, 2)}%')
print()
print(f'Mean10Yr Div: {round(mean10 * 100, 2)}%')
print(f'Median10Yr Div: {round(median10 * 100, 2)}%')
print()
print(f'Mean5Yr Div: {round(mean5 * 100, 2)}%')
print(f'Median5Yr Div: {round(median5 * 100, 2)}%')
print()
print(f'Mean3Yr Div: {round(mean3 * 100, 2)}%')
print(f'Median3Yr Div: {round(median3 * 100)}%')
print()

## DYT Gate
- The Gate is the Dividend Yield where the company is considered fair value


In [ ]:
gate = .029
gate10 = round((gate * .1) + gate, 4)
gate20 = round((gate * .20) + gate, 4)

print(f'The DYT Gate is: {round(gate * 100, 2)}%')
print(f'The DYT Gate10: {round(gate10 * 100, 2)}%')
print(f'The DYT Gate20: {round(gate20 * 100, 2)}%')

## DYT Plot

In [ ]:
dyt_fig = go.Figure([
    go.Scatter(name='DYT', y=round(dyt_df1['FwdDivYield'] * 100, 2), x=dyt_df1['Date'], mode='lines', marker_color='blue')
])
dyt_fig.add_hline(y=round(gate * 100, 2), line_dash="dash", line_color="red", annotation_text="Gate", annotation_position="right")
dyt_fig.add_hline(y=round(gate10 * 100, 2), line_dash="dash", line_color="yellow", annotation_text="Gate10", annotation_position="right")
dyt_fig.add_hline(y=round(gate20 * 100, 2), line_dash="dash", line_color="green", annotation_text="Gate20", annotation_position="right")
dyt_fig.show()

## Analysis Output

In [ ]:
metrics_json = {
    "analysis_type": "dyt",
    "analysis_date": today.strftime('%Y-%m-%d'),
    "last_df_date":dyt_df1['Date'].iloc[-1],
    "dyt_gate": gate,
    "dyt_gate10": gate10,
    "dyt_gate20": gate20,
    "mean20yr_dyt": mean20 ,
    "median20yr_dyt": median20,
    "mean10yr_dyt": mean10,
    "median10yr_dyt": median10,
    "mean5yr_dyt": median5,
    "median5yr_dyt": median5,
    "mean3yr_dyt": median3,
    "median3yr_dyt": median3
}

metrics_json

In [ ]:
metrics_df = pd.DataFrame([metrics_json])

if os.path.isfile(path_analysis_csv):
    metrics_df.to_csv(path_analysis_csv, mode='a', header=False, index=False)
else:
    metrics_df.to_csv(path_analysis_csv, mode='w', header=True, index=False)

metrics_df

## End